# Transports, Auth, and Security

In notebooks 01 to 03, the server was a local process you started yourself and could trust. Real deployments are different: servers run remotely, act for users who must sign in, and often come from third parties you do not control. Every MCP server is an access path into the systems behind it, so this notebook covers how messages travel (transports), how a remote server knows who is calling (authorization), and which controls a host and server need (security).

## Learning Objectives

By the end of this notebook, you should be able to:

- Choose between stdio and Streamable HTTP for a given server.
- Explain how a remote MCP server authorizes requests, and why it must reject tokens issued for other services.
- Match common MCP risks to the host or server control that addresses them.

## Transports: How Messages Travel

Notebook 01 split MCP into a data layer (the JSON-RPC messages) and a transport layer (how they travel). The messages are the same on every transport; only the delivery changes.

- [**stdio**](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/stdio): the client launches the server as a subprocess and exchanges one JSON message per line over its stdin and stdout. This is what the client in notebook 03 did.
- [**Streamable HTTP**](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http): the server runs as a web service. The client sends each message as an HTTP POST to a single MCP endpoint, and the reply comes back as JSON, or as a stream of events for longer responses.

| | stdio | Streamable HTTP |
|---|---|---|
| Where the server runs | Local subprocess on the user's machine | Remote web service |
| Typical use | Developer tools, local files, desktop integrations | Shared, hosted servers used by many people |
| How the caller is identified | Implicitly: it runs as the local user | With a token on every request, when the server requires authorization |
| Main risk | Inherits all of the user's local permissions | Exposed to the network, so it needs authorization and narrow scopes |

Tutorials and servers written for the first MCP version (`2024-11-05`) use a separate HTTP+SSE transport, with one endpoint for requests and a long-lived stream for responses. This is now deprecated: new remote servers should use Streamable HTTP, which can still stream responses when needed.

## Authorization For Remote Servers

How a server learns who is calling depends on the transport. A stdio server does not use MCP authorization at all: it runs as the user and reads any credentials it needs, such as an API key, from its environment. Authorization is optional in MCP. For a protected HTTP server that follows MCP authorization, the OAuth 2.1 flow has three roles:

- the **MCP server** holds the protected data and checks tokens,
- the **authorization server** signs the user in and issues tokens,
- the **MCP client** obtains a token and sends it with every request.

The flow looks like this:

```mermaid
sequenceDiagram
    participant C as MCP client
    participant U as User (browser)
    participant A as Authorization server
    participant S as MCP server
    C->>S: Request without a token
    S-->>C: 401 Unauthorized, with authorization server and scopes
    C->>U: Open the sign-in page
    U->>A: Sign in and approve the scopes
    A-->>C: Authorization code (via browser redirect)
    C->>A: Exchange the code, proving it with PKCE
    A-->>C: Access token issued for this MCP server
    C->>S: Request with the bearer token
    Note over S: Check audience, expiry, and scopes
    S-->>C: Result
```

What each step is for:

1. **The `401` tells the client where to sign in** and which scopes to ask for, so it never has to guess.
2. **The user signs in at the authorization server**, so the MCP server never sees their password. PKCE ties the sign-in to the client that started it, so an intercepted code is useless to anyone else.
3. **The token is issued for this specific MCP server**, and the client sends it as `Authorization: Bearer <token>` on every request.
4. **The server checks every request**: the token must be valid, unexpired, and issued for this server, and its scopes limit what the caller can do. A token granted `read:incidents` cannot change incident data.

Two rules prevent the most common mistakes:

- **PKCE is not authorization.** It protects the sign-in step, but the server must still validate every token and enforce its scopes.
- **No token passthrough.** A server must reject tokens issued for any other service, and must never forward the client's token to a downstream API. If Northstar needs to call `incident_db`, it uses its own credentials for that system. Otherwise a token the user granted to one service could be replayed against another, and the downstream API could not tell who is really calling.

This is the server-side *authorization* check from notebook 01, made concrete for remote servers. Approval is still separate: a valid token proves who is calling, not that a human agreed to this particular action.

## Security Controls

An MCP server is an access path: whatever it can reach, a model can reach through it. A security review asks one question per boundary in the map below.

<div align="center">
  <img src="assets/04-mcp-security-review-map.png" alt="MCP security review map" width="800" style="max-width: 100%; height: auto;" />
  <p><em>MCP security review map. Source: repository-authored diagram based on the official <a href="https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization" target="_blank" rel="noopener noreferrer">MCP authorization specification</a> and the security risks discussed in this notebook</em></p>
</div>

| Boundary | Question | Risk | Control | Enforced by |
|---|---|---|---|---|
| Server trust | Do we trust this server? | A malicious server exposes tools that misdescribe what they do or exfiltrate data. | Connect only to reviewed servers, and treat tool descriptions from unknown servers as untrusted. | Host |
| Scopes and tokens | What can it access? | A token or local process has far more access than the task needs, or a token is accepted or forwarded beyond its audience. | Request and grant least-privilege scopes, validate token audience, and use the server's own credentials downstream. | Client, authorization server, MCP server |
| Tool output | Can its output manipulate the model? | Returned content carries a prompt injection. | Treat tool results as data, never as instructions, and keep sensitive actions behind approval. | Host |
| Approval | Which actions need a human? | The model triggers a sensitive action nobody confirmed. | Pause before the call, show the tool and its arguments, and run it only after the user confirms. | Host |

The Northstar tool only returns a plan and marks critical tickets with `requires_human_approval=true`; this flag does not pause anything. A production host would review that plan and obtain approval before executing any separate sensitive action.

## Exercise: Review A Remote Incident Tool

A remote MCP server for incident management asks for a broad admin token, returns a tool result containing an instruction to reveal that token, and marks a critical escalation plan with `requires_human_approval=True`.

For each of the four boundaries (server trust, scopes and tokens, tool output, approval), name the risk in this scenario and propose a control. Say whether the host, the client, or the server enforces it. Write your answer in the cell below.

**Your answer:**

- Server trust:
- Scopes and tokens:
- Tool output:
- Approval:

<details>
<summary>Show Reference Solution</summary>

- **Server trust:** a server that asks for admin access and tries to extract tokens should not be connected at all. The host connects only to reviewed servers.
- **Scopes and tokens:** grant only the scopes the tool needs, for example `read:incidents`, never an admin token. The server validates that each token was issued for it and never forwards it to another service.
- **Tool output:** the instruction to reveal the token is untrusted data, not a command. The host must not act on it, and tokens should never be visible to the model in the first place.
- **Approval:** `requires_human_approval=True` is only a flag on a plan. The host shows the proposed action to the user and performs that separate sensitive action only after explicit confirmation.

</details>

## Check Your Understanding

1. Which transport fits a trusted helper that runs only on one developer's laptop, and why?
2. Does PKCE replace token audience validation?
3. A server receives a valid token that was issued for a different service. Should it accept the token, or forward it to the API it calls?

<details>
<summary>Show Reference Answers</summary>

1. stdio. The client launches it locally, so there is no network exposure and no OAuth flow; it reads credentials from its environment. Keep its permissions minimal, because it runs with the user's.
2. No. PKCE only protects the sign-in step. The server must still check that every token was issued for it and enforce its scopes.
3. Neither. An MCP server must only accept tokens issued for itself and must never pass client tokens on. It rejects the request with `401 Unauthorized`, and calls downstream APIs with its own credentials.

</details>

## Further Reading

- [MCP Transports](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports): How stdio and Streamable HTTP carry messages.
- [MCP Authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization): The full OAuth 2.1 flow, token validation, and scope handling.
- [Security Best Practices](https://modelcontextprotocol.io/docs/tutorials/security/security_best_practices): Attacks such as token passthrough and the confused deputy problem, with mitigations.